# 모듈 5. Human-in-the-loop 승인

기존 강의자료 `2. langgraph/1. LangGraph/3. LangGraph with Tools.ipynb`의 Tool 실행 그래프에 승인 단계를 추가합니다.

## A/B 테스트 구조

- **A: 기존 코드 그대로 사용** - 그래프가 Tool을 선택하면 바로 실행합니다.
- **B: 기존 코드에 가드레일 추가** - 위험 Tool 실행 전에 `interrupt`로 사람의 승인을 받습니다.

In [ ]:
from typing import TypedDict

from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


@tool
def refund_payment(user_id: str, amount: int, reason: str) -> str:
    """사용자에게 결제 금액을 환불한다."""
    return f"{user_id}에게 {amount:,}원 환불 완료. 사유: {reason}"


refund_request = {
    "user_id": "user-123",
    "amount": 300_000,
    "reason": "프리미엄 플랜 오결제",
}


class RefundState(TypedDict, total=False):
    user_id: str
    amount: int
    reason: str
    approval_request: dict
    approved: bool
    result: str

## 1단계: A - 기존 LangGraph Tool 실행 흐름 그대로 사용

기존 LangGraph with Tools 예제처럼 노드에서 Tool을 바로 호출합니다. 환불, 이메일, 삭제 같은 작업은 이렇게 즉시 실행되면 되돌리기 어렵습니다.

In [ ]:
def execute_refund_directly(state: RefundState) -> RefundState:
    result = refund_payment.invoke(
        {
            "user_id": state["user_id"],
            "amount": state["amount"],
            "reason": state["reason"],
        }
    )
    return {"result": result}


direct_builder = StateGraph(RefundState)
direct_builder.add_node("execute_refund", execute_refund_directly)
direct_builder.add_edge(START, "execute_refund")
direct_builder.add_edge("execute_refund", END)

direct_graph = direct_builder.compile()

print(direct_graph.invoke(refund_request)["result"])

### A안 문제점 확인

A안에서 보안 문제가 실제로 발생했는지 확인합니다.

In [ ]:
direct_state = direct_graph.invoke(refund_request)
executed_without_approval = "result" in direct_state and "approval_request" not in direct_state

print("A안 보안 진단")
print("- 승인 요청 없이 결과 생성:", executed_without_approval)
print("- 실행 결과:", direct_state["result"])
if executed_without_approval:
    print("=> 문제: 환불 같은 고위험 작업이 사람 승인 없이 실행되었습니다.")

## 2단계: B - 기존 그래프에 Human-in-the-loop Guardrail 추가

Tool 실행 노드는 유지하되, 실행 전에 승인 요청 노드를 추가합니다. `interrupt`는 그래프를 멈추고 외부 UI나 관리자 승인 값을 기다릴 수 있게 해줍니다.

In [ ]:
def plan_refund_node(state: RefundState) -> RefundState:
    return {
        "approval_request": {
            "action": "refund_payment",
            "user_id": state["user_id"],
            "amount": state["amount"],
            "reason": state["reason"],
            "risk_level": "high",
        }
    }


def approval_node(state: RefundState) -> RefundState:
    decision = interrupt(state["approval_request"])
    approved = bool(decision.get("approved")) if isinstance(decision, dict) else bool(decision)
    return {"approved": approved}


def execute_refund_after_approval(state: RefundState) -> RefundState:
    if not state.get("approved"):
        return {"result": "중단: 사람이 환불 요청을 승인하지 않았습니다."}
    result = refund_payment.invoke(
        {
            "user_id": state["user_id"],
            "amount": state["amount"],
            "reason": state["reason"],
        }
    )
    return {"result": result}


approval_builder = StateGraph(RefundState)
approval_builder.add_node("plan_refund", plan_refund_node)
approval_builder.add_node("approval", approval_node)
approval_builder.add_node("execute_refund", execute_refund_after_approval)
approval_builder.add_edge(START, "plan_refund")
approval_builder.add_edge("plan_refund", "approval")
approval_builder.add_edge("approval", "execute_refund")
approval_builder.add_edge("execute_refund", END)

approval_graph = approval_builder.compile(checkpointer=MemorySaver())

### B안 개선 확인

B안에서 기존 코드 흐름은 유지하면서 보안 문제가 줄어드는지 확인합니다.

In [ ]:
verify_config = {"configurable": {"thread_id": "refund-approval-diagnostic"}}
paused_state = approval_graph.invoke(refund_request, config=verify_config)
has_interrupt = "__interrupt__" in paused_state

print("B안 보안 진단")
print("- 실행 전 interrupt 발생:", has_interrupt)
print("- 승인 요청 payload:", paused_state["__interrupt__"][0].value)
print("=> 개선: 위험 Tool 실행 전에 사람 승인 지점이 생겼습니다.")

In [ ]:
config = {"configurable": {"thread_id": "refund-approval-demo"}}

first_result = approval_graph.invoke(refund_request, config=config)
print("승인 요청:")
print(first_result["__interrupt__"][0].value)

approved_result = approval_graph.invoke(
    Command(resume={"approved": True}),
    config=config,
)
print("\n승인 후 결과:")
print(approved_result["result"])

## 정리

기존 LangGraph Tool 실행 그래프는 그대로 두고, 위험 Tool 앞에 승인 노드를 추가했습니다. Human-in-the-loop는 모든 작업에 붙이는 것이 아니라 환불, 삭제, 외부 발송처럼 되돌리기 어려운 작업에 선별 적용합니다.